# Naive Bayes & NLP — Applied on Real Dataset
**20 Newsgroups Dataset pe Text Classification**

---
## Table of Contents
1. Dataset Overview — 20 Newsgroups
2. Data Exploration — Text Analysis
3. Text Preprocessing Pipeline
4. Model Building — MultinomialNB
5. Model Evaluation — Classification Report
6. Hyperparameter Tuning — Alpha (Laplace Smoothing)
7. Capstone Challenge + Solutions
---

## Section 1: Dataset Overview

### Keywords:
- **20 Newsgroups:** 18,000+ newsgroup posts, 20 categories
- **Text Classification:** Automatically assign category to a text
- **Corpus:** Poori collection of text documents
- **Category/Label:** Har newsgroup post ka topic (sports, tech, religion, etc.)
- **`fetch_20newsgroups`:** Sklearn mein built-in dataset function

In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np
import matplotlib.pyplot as plt

# 4 categories select karte hain (puri 20 categories slow hoti hain)
categories = [
    'rec.sport.cricket',
    'sci.med',
    'talk.politics.guns',
    'comp.graphics',
]

# Load train and test
train_data = fetch_20newsgroups(
    subset='train', categories=categories,
    remove=('headers', 'footers', 'quotes'),  # remove metadata
    random_state=42
)
test_data = fetch_20newsgroups(
    subset='test', categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

print(f"Training samples: {len(train_data.data)}")
print(f"Testing samples:  {len(test_data.data)}")
print(f"Categories: {train_data.target_names}")
print(f"\n--- Sample Post (Category: {train_data.target_names[train_data.target[0]]}) ---")
print(train_data.data[0][:400], "...")

In [ ]:
# Class distribution
unique, counts = np.unique(train_data.target, return_counts=True)

plt.figure(figsize=(8, 4))
plt.bar([train_data.target_names[i] for i in unique], counts, color='steelblue', alpha=0.8)
plt.xticks(rotation=15, ha='right')
plt.ylabel('Number of Posts')
plt.title('Training Data — Class Distribution')
plt.tight_layout()
plt.show()

print("Class distribution:")
for i, cnt in zip(unique, counts):
    print(f"  {train_data.target_names[i]:<30} {cnt} posts")

## Section 2: Text Preprocessing + Feature Extraction

### Keywords:
- **TF-IDF:** Term Frequency × Inverse Document Frequency — common words ko less weight deta hai
- **`max_features`:** Kitne unique words vocabulary mein rakhne hain
- **`min_df`:** Minimum document frequency — bahut rare words hata deta hai
- **`max_df`:** Maximum document frequency — bahut common words hata deta hai (stop words jaise)
- **`ngram_range`:** (1,2) means unigrams (single words) + bigrams (word pairs)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Pipeline: TF-IDF → Naive Bayes
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        min_df=2,          # word must appear in at least 2 docs
        max_df=0.95,       # ignore words in >95% of docs
        ngram_range=(1, 2) # unigrams + bigrams
    )),
    ('nb', MultinomialNB(alpha=0.1))
])

# Train
pipeline.fit(train_data.data, train_data.target)

# Check vocabulary size
vocab_size = len(pipeline.named_steps['tfidf'].vocabulary_)
print(f"Vocabulary size: {vocab_size} features")
print(f"Training samples: {len(train_data.data)}")
print(f"\nFeature matrix shape: documents × features")
X_train_transformed = pipeline.named_steps['tfidf'].transform(train_data.data)
print(f"  {X_train_transformed.shape}")
print(f"  (Sparse matrix — most values are 0)")

## Section 3: Model Evaluation

### Keywords:
- **Classification Report:** Precision, Recall, F1-score per class
- **Precision:** Jab model ne positive kaha, kitni baar sahi tha?
- **Recall:** Actual positives mein se kitne pakde?
- **F1-Score:** Precision aur Recall ka harmonic mean
- **Confusion Matrix:** Kaunsa class kaunse class se confuse ho raha hai

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Predict
y_pred = pipeline.predict(test_data.data)
y_true = test_data.target

# Classification report
print("=== Classification Report ===")
print(classification_report(
    y_true, y_pred,
    target_names=test_data.target_names
))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_estimator(
    pipeline, test_data.data, y_true,
    display_labels=[c.split('.')[-1] for c in test_data.target_names],
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix — Naive Bayes on 20 Newsgroups')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Most informative features per class
tfidf = pipeline.named_steps['tfidf']
nb_model = pipeline.named_steps['nb']
feature_names = np.array(tfidf.get_feature_names_out())

print("=== Top Words per Category ===")
for i, category in enumerate(train_data.target_names):
    # Log probability of each feature given this class
    top_indices = nb_model.feature_log_prob_[i].argsort()[-10:][::-1]
    top_words = feature_names[top_indices]
    print(f"\n{category.split('.')[-1].upper()}:")
    print(', '.join(top_words))

## Section 4: Hyperparameter Tuning — Alpha

### Keywords:
- **Laplace Smoothing (alpha):** Zero probability problem solve karta hai
  - alpha=0: No smoothing (can give 0 probability = problem)
  - alpha=1: Standard Laplace smoothing (default)
  - alpha>1: More smoothing (underfits)
  - alpha<1: Less smoothing (can overfit)
- **GridSearchCV:** Automatically best hyperparameters find karta hai

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'tfidf__max_features': [5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'nb__alpha': [0.01, 0.1, 0.5, 1.0]
}

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(train_data.data, train_data.target)

print(f"\nBest Parameters: {grid.best_params_}")
print(f"Best CV Accuracy: {grid.best_score_:.3f}")

# Test with best model
best_acc = grid.score(test_data.data, test_data.target)
print(f"Test Accuracy (best model): {best_acc:.3f}")

In [ ]:
# Alpha effect visualization
alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0]
train_scores = []
test_scores = []

for alpha in alphas:
    p = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000)),
        ('nb', MultinomialNB(alpha=alpha))
    ])
    p.fit(train_data.data, train_data.target)
    train_scores.append(p.score(train_data.data, train_data.target))
    test_scores.append(p.score(test_data.data, test_data.target))

plt.figure(figsize=(8, 4))
plt.semilogx(alphas, train_scores, 'b-o', label='Train Accuracy')
plt.semilogx(alphas, test_scores, 'r-o', label='Test Accuracy')
plt.xlabel('Alpha (log scale)')
plt.ylabel('Accuracy')
plt.title('Effect of Alpha (Laplace Smoothing)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_alpha = alphas[np.argmax(test_scores)]
print(f"Best alpha for test accuracy: {best_alpha}")

## Section 5: Live Prediction Demo

In [ ]:
# Predict new texts
best_model = grid.best_estimator_

new_texts = [
    "The patient showed symptoms of fever and was given antibiotics by the doctor",
    "The batsman hit a brilliant century in the test match against Australia",
    "The new graphics card supports ray tracing and 4K resolution gaming",
    "The senator proposed new regulations on gun ownership and background checks"
]

predictions = best_model.predict(new_texts)
probabilities = best_model.predict_proba(new_texts)

print("=== Live Predictions ===")
for text, pred, probs in zip(new_texts, predictions, probabilities):
    pred_class = train_data.target_names[pred].split('.')[-1]
    confidence = probs.max() * 100
    print(f"\nText: {text[:60]}...")
    print(f"  Predicted: {pred_class.upper()} ({confidence:.1f}% confidence)")
    print(f"  All probabilities:")
    for cat, p in zip([c.split('.')[-1] for c in train_data.target_names], probs):
        bar = '█' * int(p * 20)
        print(f"    {cat:<10} {p:.3f} {bar}")

## Section 6: Aaj Humne Kya Seekha?

### Summary:
1. **20 Newsgroups dataset** — Real-world text classification problem
2. **TF-IDF Pipeline** — Text → Numerical features automatically
3. **Classification Report** — Precision, Recall, F1 per class
4. **Alpha tuning** — Too low = overfit, too high = underfit
5. **Bigrams** — Word pairs capture context better than single words

### Real-World Uses:
- **Email Spam Filter** — Gmail, Outlook
- **News Article Categorization** — Automatic topic tagging
- **Customer Support Routing** — Automatically route tickets to right team
- **Sentiment Analysis** — Product reviews positive/negative
- **Language Detection** — Identify language of text

---
## Capstone Challenge

**Problem:** SMS Spam Detection using the UCI SMS Spam Collection Dataset.

Use this inline dataset (no download needed):

**Tasks:**
1. Create a labeled SMS dataset with at least 20 messages (10 spam, 10 ham)
2. Build a TF-IDF + MultinomialNB pipeline
3. Split 70/30 train/test and evaluate
4. Show the classification report
5. Try alpha values [0.01, 0.1, 1.0] and plot their test accuracy
6. Predict 3 new custom messages you write yourself

In [ ]:
# CAPSTONE — TRY YOURSELF FIRST!
# Scroll down for solutions

# Sample messages to get you started
messages = [
    "WINNER!! You have been selected for a cash prize of Rs. 50000. Call now!",
    "Hey, are you coming to the meeting at 3pm today?",
    "FREE entry to win Rs 1,00,000 prize. Text WIN to 80809",
    "Can you pick up some groceries on your way home?",
    "Congratulations! Your mobile number has WON a free vacation. Click here",
    # Add more...
]
labels = ['spam', 'ham', 'spam', 'ham', 'spam']

print("Your turn! Complete the capstone challenge...")
# YOUR CODE HERE


---
## Solutions — Capstone

In [ ]:
# === SOLUTION ===
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt

# Task 1: 20-message dataset
messages = [
    "WINNER!! You have been selected for a cash prize of Rs. 50000. Call now!",
    "Hey, are you coming to the meeting at 3pm today?",
    "FREE entry to win Rs 1,00,000 prize. Text WIN to 80809",
    "Can you pick up some groceries on your way home?",
    "Congratulations! Your mobile number has WON a free vacation. Click here",
    "What time are we meeting tomorrow for lunch?",
    "URGENT: Your account has been compromised. Verify now at this link",
    "Happy birthday! Hope you have a wonderful day",
    "You have been pre-approved for a loan of Rs 10 lakh. No credit check!",
    "Did you watch the match last night? Amazing innings!",
    "Act now! Limited offer — 90% discount on iPhone. Only today!",
    "I'll be late by 15 minutes, traffic is heavy",
    "Earn Rs 5000 per day working from home. No experience needed. Register now",
    "Can you send me the notes from yesterday's lecture?",
    "Your OTP is 847291. Do not share with anyone",
    "See you at the library at 5pm for the study session",
    "Claim your FREE gift now! This offer expires in 24 hours. Reply YES",
    "Thank you for helping me yesterday, really appreciate it",
    "You are selected for a secret investment opportunity. Guaranteed returns!",
    "Running a bit late, save me a seat please"
]
labels = ['spam','ham','spam','ham','spam','ham','spam','ham','spam','ham',
          'spam','ham','spam','ham','ham','ham','spam','ham','spam','ham']

# Task 2: Build pipeline
X_train, X_test, y_train, y_test = train_test_split(
    messages, labels, test_size=0.3, random_state=42, stratify=labels
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB(alpha=0.1))
])
pipeline.fit(X_train, y_train)

# Task 3+4: Evaluate
y_pred = pipeline.predict(X_test)
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

# Task 5: Alpha comparison
alphas = [0.01, 0.1, 1.0]
test_accs = []
for alpha in alphas:
    p = Pipeline([('tfidf', TfidfVectorizer()), ('nb', MultinomialNB(alpha=alpha))])
    p.fit(X_train, y_train)
    test_accs.append(p.score(X_test, y_test))

plt.figure(figsize=(6, 4))
plt.bar([str(a) for a in alphas], test_accs, color='steelblue', alpha=0.8)
for i, acc in enumerate(test_accs):
    plt.text(i, acc + 0.01, f'{acc:.2f}', ha='center')
plt.xlabel('Alpha')
plt.ylabel('Test Accuracy')
plt.title('Alpha vs Accuracy')
plt.ylim(0, 1.1)
plt.show()

# Task 6: Custom predictions
custom = [
    "You won a lottery of Rs 2 lakh. Claim now!",
    "See you at the coffee shop at 6pm",
    "Free iPhone for first 100 callers, dial now!"
]
preds = pipeline.predict(custom)
probs = pipeline.predict_proba(custom)
print("\n=== Custom Predictions ===")
for msg, pred, prob in zip(custom, preds, probs):
    conf = prob.max() * 100
    print(f"Message: {msg[:50]}")
    print(f"  → {pred.upper()} ({conf:.0f}% confidence)\n")